# 🚦 Event-Driven Congestion Prediction & Resource Planning
### Hackathon Notebook — Planned & Unplanned Traffic Disruptions

**Problem:** Political rallies, festivals, sports events, construction, breakdowns and sudden gatherings create localized traffic breakdowns.  
**Goal:** Forecast event-related traffic impact and recommend optimal manpower, barricading, and diversion plans.

---

## Pipeline Overview
```
Raw CSV → EDA → Feature Engineering → ML Models → Impact Score → Resource Recommendation
```

Models used:
- **LightGBM** — Duration (hours) regression
- **XGBoost** — Road closure classification
- **Ensemble** — Impact severity score
- **Rule-based heuristic** — Manpower / barricade / diversion recommendation

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    mean_absolute_error, r2_score,
    classification_report, roc_auc_score, confusion_matrix
)
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import Pipeline

import lightgbm as lgb
import xgboost as xgb

from scipy.stats import chi2_contingency

# Aesthetics
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.titlesize'] = 13

SEED = 42
print('✅ Libraries loaded.')

## 2. Load Data

In [ ]:
# ── CHANGE THIS PATH if running locally ──
CSV_PATH = '/mnt/user-data/uploads/Astram_event_data_anonymized_-_Astram_event_data_anonymizedb40ac87.csv'

df_raw = pd.read_csv(CSV_PATH)
print(f'Dataset shape: {df_raw.shape}')
df_raw.head(3)

## 3. Exploratory Data Analysis

### 3.1 Basic Overview

In [ ]:
print('=== DATA TYPES ===')
print(df_raw.dtypes)
print('\n=== MISSING VALUES (top 20) ===')
missing = (df_raw.isnull().sum() / len(df_raw) * 100).sort_values(ascending=False)
print(missing[missing > 0].head(20).round(1).astype(str) + '%')

In [ ]:
# Missingness heatmap (sampled)
plt.figure(figsize=(14, 4))
sns.heatmap(
    df_raw[missing[missing > 0].index].sample(300, random_state=SEED).isnull(),
    cbar=False, yticklabels=False, cmap='YlOrRd'
)
plt.title('Missing Value Heatmap (300-row sample)')
plt.tight_layout()
plt.show()

### 3.2 Event Type & Cause Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Event type
vc = df_raw['event_type'].value_counts()
axes[0].bar(vc.index, vc.values, color=['#4C72B0', '#DD8452'])
axes[0].set_title('Event Type Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(vc.values):
    axes[0].text(i, v + 50, str(v), ha='center', fontweight='bold')

# Event cause
ec = df_raw['event_cause'].value_counts()
ec.plot(kind='barh', ax=axes[1], color=sns.color_palette('muted', len(ec)))
axes[1].set_title('Event Cause Distribution')
axes[1].set_xlabel('Count')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print('\nKey observation: ~60% vehicle breakdowns (unplanned). Public events, processions, protests = planned disruptions.')

### 3.3 Temporal Patterns

In [ ]:
df_raw['start_dt'] = pd.to_datetime(df_raw['start_datetime'], utc=True, errors='coerce')
df_raw['hour'] = df_raw['start_dt'].dt.hour
df_raw['dow']  = df_raw['start_dt'].dt.day_name()
df_raw['month'] = df_raw['start_dt'].dt.month

DOW_ORDER = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Hour of day
hour_counts = df_raw['hour'].value_counts().sort_index()
axes[0].bar(hour_counts.index, hour_counts.values, color='steelblue')
axes[0].set_title('Events by Hour of Day')
axes[0].set_xlabel('Hour (UTC)')
axes[0].set_ylabel('Count')

# Day of week
dow_counts = df_raw['dow'].value_counts().reindex(DOW_ORDER)
axes[1].bar(DOW_ORDER, dow_counts.values, color='coral')
axes[1].set_title('Events by Day of Week')
axes[1].tick_params(axis='x', rotation=30)

# Month
month_map = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',11:'Nov',12:'Dec'}
mc = df_raw['month'].value_counts().sort_index()
axes[2].bar([month_map.get(m,m) for m in mc.index], mc.values, color='mediumseagreen')
axes[2].set_title('Events by Month')

plt.tight_layout()
plt.show()

In [ ]:
# Planned vs Unplanned — hour comparison
fig, ax = plt.subplots(figsize=(14, 5))
for etype, color in [('planned','#DD8452'), ('unplanned','#4C72B0')]:
    h = df_raw[df_raw['event_type'] == etype]['hour'].value_counts().sort_index()
    ax.plot(h.index, h.values, marker='o', label=etype.title(), color=color)
ax.set_title('Planned vs Unplanned — Hour of Day Profile')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Incident Count')
ax.legend()
plt.tight_layout()
plt.show()

### 3.4 Spatial Analysis — Hotspot Zones

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Zone distribution
zone_vc = df_raw['zone'].value_counts().head(10)
zone_vc.plot(kind='barh', ax=axes[0], color='mediumpurple')
axes[0].set_title('Top 10 Zones by Incident Count')
axes[0].invert_yaxis()

# Top junctions
junc_vc = df_raw['junction'].value_counts().head(15)
junc_vc.plot(kind='barh', ax=axes[1], color='tomato')
axes[1].set_title('Top 15 Junctions by Incident Count')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# 2D spatial scatter (lat/lon)
plt.figure(figsize=(10, 8))
planned   = df_raw[df_raw['event_type'] == 'planned']
unplanned = df_raw[df_raw['event_type'] == 'unplanned']

plt.scatter(unplanned['longitude'], unplanned['latitude'],
            alpha=0.15, s=8, color='steelblue', label='Unplanned')
plt.scatter(planned['longitude'], planned['latitude'],
            alpha=0.6, s=20, color='red', label='Planned', marker='^')

plt.title('Incident Spatial Distribution — Bengaluru')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.legend()
plt.tight_layout()
plt.show()

### 3.5 Priority, Road Closure & Duration

In [ ]:
df_raw['closed_dt']   = pd.to_datetime(df_raw['closed_datetime'], utc=True, errors='coerce')
df_raw['resolved_dt'] = pd.to_datetime(df_raw['resolved_datetime'], utc=True, errors='coerce')
df_raw['effective_close'] = df_raw['closed_dt'].fillna(df_raw['resolved_dt'])
df_raw['duration_hours'] = (
    (df_raw['effective_close'] - df_raw['start_dt']).dt.total_seconds() / 3600
)
# Clip to reasonable range [0, 200]
df_dur = df_raw[(df_raw['duration_hours'] > 0) & (df_raw['duration_hours'] < 200)].copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Duration by event cause
cause_dur = df_dur.groupby('event_cause')['duration_hours'].median().sort_values(ascending=False)
cause_dur.plot(kind='barh', ax=axes[0], color='teal')
axes[0].set_title('Median Duration (hrs) by Event Cause')
axes[0].invert_yaxis()

# Road closure by cause
rc_rate = (
    df_raw.groupby('event_cause')['requires_road_closure']
    .mean().sort_values(ascending=False) * 100
)
rc_rate.plot(kind='barh', ax=axes[1], color='darkorange')
axes[1].set_title('Road Closure Rate (%) by Cause')
axes[1].invert_yaxis()

# Duration distribution
axes[2].hist(df_dur['duration_hours'], bins=50, color='slateblue', edgecolor='white')
axes[2].set_title('Distribution of Resolution Duration')
axes[2].set_xlabel('Hours')

plt.tight_layout()
plt.show()

In [ ]:
# Priority vs Corridor heatmap
pivot = pd.crosstab(df_raw['priority'], df_raw['event_cause'])
plt.figure(figsize=(14, 4))
sns.heatmap(pivot, annot=True, fmt='d', cmap='YlOrRd', linewidths=0.5)
plt.title('Priority vs Event Cause (count)')
plt.tight_layout()
plt.show()

## 4. Feature Engineering

In [ ]:
df = df_raw.copy()

# ── Temporal features ──
df['hour']          = df['start_dt'].dt.hour
df['dow_num']       = df['start_dt'].dt.dayofweek      # 0=Mon
df['month']         = df['start_dt'].dt.month
df['is_weekend']    = (df['dow_num'] >= 5).astype(int)
df['is_peak_am']    = df['hour'].between(6, 9).astype(int)   # 6–9 AM
df['is_peak_pm']    = df['hour'].between(17, 20).astype(int) # 5–8 PM
df['is_nighttime']  = (df['hour'] <= 5).astype(int)

# ── Spatial features ──
# Distance from city centre (MG Road ~12.9715, 77.5946)
CITY_LAT, CITY_LON = 12.9715987, 77.5945627
df['dist_from_centre'] = np.sqrt(
    (df['latitude'] - CITY_LAT)**2 + (df['longitude'] - CITY_LON)**2
) * 111  # approx km

# Is corridor (high-traffic road)?
df['is_corridor'] = (~df['corridor'].isin(['Non-corridor', np.nan, None])).astype(int)

# Junction available?
df['has_junction'] = df['junction'].notna().astype(int)

# ── Event metadata features ──
df['is_planned'] = (df['event_type'] == 'planned').astype(int)

# Group event causes into broader categories
PUBLIC_EVENTS = {'public_event','processional','processional','protest','vip_movement','procession'}
INFRA_ISSUES  = {'pot_holes','water_logging','construction','road_conditions','Debris','debris'}
VEHICLE_ISSUES = {'vehicle_breakdown','accident'}
NATURAL       = {'tree_fall', 'Fog / Low Visibility'}

def categorize_cause(cause):
    if pd.isna(cause): return 'unknown'
    if cause in PUBLIC_EVENTS:  return 'public_event'
    if cause in INFRA_ISSUES:   return 'infrastructure'
    if cause in VEHICLE_ISSUES: return 'vehicle'
    if cause in NATURAL:        return 'natural'
    return 'others'

df['cause_category'] = df['event_cause'].apply(categorize_cause)

# Historical hotspot score: junction-level incident frequency
junc_freq = df['junction'].value_counts()
df['junction_hotspot_score'] = df['junction'].map(junc_freq).fillna(0)

# Zone-level baseline risk
zone_freq = df['zone'].value_counts()
df['zone_risk_score'] = df['zone'].map(zone_freq).fillna(0)

# Priority encoding
df['priority_num'] = df['priority'].map({'High': 1, 'Low': 0}).fillna(0)

print('Feature engineering complete.')
print(f'New columns: hour, dow_num, month, is_weekend, is_peak_am, is_peak_pm, is_nighttime, '
      f'dist_from_centre, is_corridor, has_junction, is_planned, cause_category, '
      f'junction_hotspot_score, zone_risk_score, priority_num')

In [ ]:
# Encode categoricals for ML
cat_cols = ['event_cause', 'event_type', 'corridor', 'zone', 'cause_category',
            'police_station', 'gba_identifier']

le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col + '_enc'] = le.fit_transform(df[col].fillna('unknown').astype(str))
    le_dict[col] = le

print('Label encoding done.')

## 5. Model 1 — Duration Regression (LightGBM)
**Target:** `duration_hours` — how long will this incident disrupt traffic?

> This tells ops teams *how long* to deploy resources.

In [ ]:
FEAT_DUR = [
    'hour', 'dow_num', 'month', 'is_weekend', 'is_peak_am', 'is_peak_pm', 'is_nighttime',
    'dist_from_centre', 'is_corridor', 'has_junction', 'is_planned', 'priority_num',
    'junction_hotspot_score', 'zone_risk_score',
    'event_cause_enc', 'corridor_enc', 'zone_enc', 'cause_category_enc',
    'latitude', 'longitude'
]

# Filter: only rows with valid duration (0–200 hrs)
mask = (df['duration_hours'] > 0) & (df['duration_hours'] < 200)
df_dur_ml = df[mask].copy()
print(f'Duration model dataset: {len(df_dur_ml)} rows')

X_dur = df_dur_ml[FEAT_DUR]
y_dur = np.log1p(df_dur_ml['duration_hours'])  # log-transform for skew

X_tr, X_te, y_tr, y_te = train_test_split(X_dur, y_dur, test_size=0.2, random_state=SEED)

lgb_model = lgb.LGBMRegressor(
    n_estimators=600,
    learning_rate=0.05,
    num_leaves=63,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=SEED,
    verbose=-1
)

lgb_model.fit(X_tr, y_tr,
              eval_set=[(X_te, y_te)],
              callbacks=[lgb.early_stopping(50, verbose=False),
                         lgb.log_evaluation(period=-1)])

y_pred_log = lgb_model.predict(X_te)
y_pred_dur = np.expm1(y_pred_log)
y_true_dur = np.expm1(y_te)

mae  = mean_absolute_error(y_true_dur, y_pred_dur)
r2   = r2_score(y_te, y_pred_log)
print(f'\n📊 LightGBM Duration Model Results:')
print(f'   MAE  : {mae:.2f} hours')
print(f'   R²   : {r2:.3f}')
print(f'   Median predicted: {np.median(y_pred_dur):.2f} hrs, Actual: {np.median(y_true_dur):.2f} hrs')

In [ ]:
# Feature importance
fi = pd.Series(lgb_model.feature_importances_, index=FEAT_DUR).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, 6))
fi.head(15).plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('LightGBM — Feature Importance (Duration Regression)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Predicted vs Actual scatter
plt.figure(figsize=(7, 6))
plt.scatter(y_true_dur, y_pred_dur, alpha=0.3, s=10, color='steelblue')
maxv = min(max(y_true_dur.max(), y_pred_dur.max()), 150)
plt.plot([0, maxv], [0, maxv], 'r--', lw=1.5, label='Perfect prediction')
plt.xlabel('Actual Duration (hrs)')
plt.ylabel('Predicted Duration (hrs)')
plt.title('Duration: Predicted vs Actual')
plt.xlim(0, maxv); plt.ylim(0, maxv)
plt.legend()
plt.tight_layout()
plt.show()

## 6. Model 2 — Road Closure Classification (XGBoost)
**Target:** `requires_road_closure` (binary: 0/1)

> A positive prediction triggers barricading & diversion deployment.

In [ ]:
FEAT_CLS = [
    'hour', 'dow_num', 'month', 'is_weekend', 'is_peak_am', 'is_peak_pm', 'is_nighttime',
    'dist_from_centre', 'is_corridor', 'has_junction', 'is_planned', 'priority_num',
    'junction_hotspot_score', 'zone_risk_score',
    'event_cause_enc', 'corridor_enc', 'zone_enc', 'cause_category_enc',
    'latitude', 'longitude'
]

df_cls = df.dropna(subset=['requires_road_closure'])
X_cls = df_cls[FEAT_CLS]
y_cls = df_cls['requires_road_closure'].astype(int)

print(f'Dataset: {len(df_cls)} rows | Class balance: {y_cls.mean():.2%} road closures')

X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X_cls, y_cls, test_size=0.2,
                                                stratify=y_cls, random_state=SEED)

# Class weight to handle imbalance
neg, pos = (y_tr2 == 0).sum(), (y_tr2 == 1).sum()
scale_pos = neg / pos

xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos,
    reg_alpha=0.1,
    reg_lambda=1.0,
    use_label_encoder=False,
    eval_metric='auc',
    random_state=SEED,
    verbosity=0
)

xgb_model.fit(X_tr2, y_tr2,
              eval_set=[(X_te2, y_te2)],
              verbose=False)

y_prob = xgb_model.predict_proba(X_te2)[:, 1]
y_pred_cls = (y_prob >= 0.4).astype(int)  # tuned threshold

print('\n📊 XGBoost Road Closure Classifier Results:')
print(f'   ROC-AUC: {roc_auc_score(y_te2, y_prob):.3f}')
print()
print(classification_report(y_te2, y_pred_cls, target_names=['No Closure','Road Closure']))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_te2, y_pred_cls)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Closure','Closure'],
            yticklabels=['No Closure','Closure'])
plt.title('Confusion Matrix — Road Closure Classifier')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

In [ ]:
# XGBoost feature importance
fi2 = pd.Series(xgb_model.feature_importances_, index=FEAT_CLS).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, 6))
fi2.head(15).plot(kind='barh', ax=ax, color='darkorange')
ax.set_title('XGBoost — Feature Importance (Road Closure)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 7. Ensemble — Traffic Impact Severity Score

Combine both models into a single **Impact Score [0–10]** used for prioritization and resource allocation.

In [ ]:
# Re-run on a common test set (use df_cls filtered rows that also have duration)
df_joint = df[(mask) & (df['requires_road_closure'].notna())].copy()
print(f'Joint dataset: {len(df_joint)} rows')

# Predict durations
X_joint = df_joint[FEAT_DUR]
dur_preds = np.expm1(lgb_model.predict(X_joint))
dur_preds = np.clip(dur_preds, 0, 200)

# Predict road closure prob
X_joint_cls = df_joint[FEAT_CLS]
closure_probs = xgb_model.predict_proba(X_joint_cls)[:, 1]

# Normalize duration to 0-10
dur_score = np.clip(dur_preds / 20, 0, 10)  # 20 hrs → score 10

# Road closure score
closure_score = closure_probs * 10

# Priority bonus
priority_bonus = df_joint['priority_num'].values * 1.5

# Hotspot weight
hotspot_score = np.clip(df_joint['junction_hotspot_score'].values / 20, 0, 2.0)

# Weighted ensemble
IMPACT_SCORE = np.clip(
    0.35 * dur_score +
    0.35 * closure_score +
    0.15 * priority_bonus +
    0.15 * hotspot_score,
    0, 10
)

df_joint['predicted_duration_hrs'] = dur_preds
df_joint['closure_probability']    = closure_probs
df_joint['impact_score']           = IMPACT_SCORE

print('\n📊 Impact Score Distribution:')
print(pd.Series(IMPACT_SCORE).describe().round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(IMPACT_SCORE, bins=40, color='crimson', edgecolor='white')
axes[0].set_title('Distribution of Traffic Impact Scores')
axes[0].set_xlabel('Impact Score (0–10)')

# Impact by cause category
df_joint['cause_category_col'] = df_joint['event_cause'].apply(categorize_cause)
cause_impact = df_joint.groupby('cause_category_col')['impact_score'].median().sort_values(ascending=False)
cause_impact.plot(kind='bar', ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('Median Impact Score by Cause Category')
axes[1].set_ylabel('Impact Score')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

## 8. Resource Recommendation Engine

Given a predicted impact score + closure probability + duration, generate a recommended deployment plan.

In [ ]:
def recommend_resources(impact_score, closure_prob, predicted_hours, event_cause, is_planned):
    """
    Rule-based resource recommendation engine.
    
    Returns a dict with:
      - manpower          : number of personnel to deploy
      - barricades        : number of barricades
      - diversion_routes  : number of alternate routes to activate
      - patrol_frequency  : how often to check (in minutes)
      - alert_level       : 'LOW' / 'MEDIUM' / 'HIGH' / 'CRITICAL'
      - action_notes      : human-readable instructions
    """
    # Thresholds for alert levels
    if impact_score >= 7.5:
        level = 'CRITICAL'
    elif impact_score >= 5.0:
        level = 'HIGH'
    elif impact_score >= 2.5:
        level = 'MEDIUM'
    else:
        level = 'LOW'

    # Base manpower
    manpower_map = {'LOW': 2, 'MEDIUM': 5, 'HIGH': 10, 'CRITICAL': 20}
    manpower = manpower_map[level]

    # Closure adds more
    if closure_prob > 0.6:
        manpower = int(manpower * 1.5)

    # Long events need more sustained deployment
    if predicted_hours > 6:
        manpower += 3

    # Barricades
    barricades_map = {'LOW': 0, 'MEDIUM': 4, 'HIGH': 10, 'CRITICAL': 20}
    barricades = barricades_map[level]
    if closure_prob > 0.5:
        barricades = max(barricades, int(closure_prob * 25))

    # Diversion routes
    diversion_map = {'LOW': 0, 'MEDIUM': 1, 'HIGH': 2, 'CRITICAL': 3}
    diversion_routes = diversion_map[level]

    # Patrol frequency
    patrol_map = {'LOW': 60, 'MEDIUM': 30, 'HIGH': 15, 'CRITICAL': 10}
    patrol_frequency = patrol_map[level]

    # Action notes
    notes = []
    if level in ('HIGH', 'CRITICAL'):
        notes.append('⚠️  Notify Traffic Control Centre immediately.')
    if closure_prob > 0.5:
        notes.append('🚧 Pre-position barricades at both ends of affected corridor.')
    if predicted_hours > 4:
        notes.append(f'🕐 Plan for sustained deployment (~{predicted_hours:.1f} hrs). Schedule shift rotation.')
    if event_cause in ('public_event', 'processional', 'procession'):
        notes.append('📢 Coordinate with event organizers for procession path and timing.')
    if event_cause == 'construction':
        notes.append('🏗️  Liaise with contractor for work hours — enforce nighttime window if possible.')
    if event_cause in ('accident', 'vehicle_breakdown'):
        notes.append('🚑  Coordinate with emergency services; tow truck deployment recommended.')
    if is_planned:
        notes.append('📋 Pre-deploy resources 2 hours before event start.')
    if not notes:
        notes.append('✅ Monitor situation. Standard patrolling protocol.')

    return {
        'alert_level'      : level,
        'manpower'         : manpower,
        'barricades'       : barricades,
        'diversion_routes' : diversion_routes,
        'patrol_freq_min'  : patrol_frequency,
        'action_notes'     : notes
    }


# ── Test the recommender ──
test_cases = [
    dict(impact_score=8.2, closure_prob=0.85, predicted_hours=12, event_cause='public_event',  is_planned=True),
    dict(impact_score=5.5, closure_prob=0.60, predicted_hours=4,  event_cause='construction',  is_planned=False),
    dict(impact_score=3.1, closure_prob=0.20, predicted_hours=1.5,event_cause='vehicle_breakdown', is_planned=False),
    dict(impact_score=1.2, closure_prob=0.05, predicted_hours=0.5,event_cause='others',        is_planned=False),
]

print('=' * 60)
for i, tc in enumerate(test_cases, 1):
    rec = recommend_resources(**tc)
    print(f'\n🔹 Test Case {i} | Cause: {tc["event_cause"]} | Impact: {tc["impact_score"]}')
    print(f'   Alert Level    : {rec["alert_level"]}')
    print(f'   Manpower       : {rec["manpower"]} personnel')
    print(f'   Barricades     : {rec["barricades"]} units')
    print(f'   Diversion Routes: {rec["diversion_routes"]}')
    print(f'   Patrol Every   : {rec["patrol_freq_min"]} min')
    print('   Notes:')
    for note in rec['action_notes']:
        print(f'     {note}')
print('\n' + '=' * 60)

## 9. Apply to Full Dataset — Ranked Incident Table

In [ ]:
# Predict on ALL rows (fill missing duration with median from same cause)
X_full = df[FEAT_DUR].copy()
dur_all = np.expm1(lgb_model.predict(X_full))

X_full_cls = df[FEAT_CLS].copy()
prob_all = xgb_model.predict_proba(X_full_cls)[:, 1]

dur_score_all   = np.clip(dur_all / 20, 0, 10)
closure_all     = prob_all * 10
priority_all    = df['priority_num'].values * 1.5
hotspot_all     = np.clip(df['junction_hotspot_score'].values / 20, 0, 2.0)

impact_all = np.clip(
    0.35 * dur_score_all +
    0.35 * closure_all +
    0.15 * priority_all +
    0.15 * hotspot_all, 0, 10
)

df['predicted_duration_hrs'] = np.clip(dur_all, 0, 200)
df['closure_prob']           = prob_all
df['impact_score']           = impact_all

# Apply recommender
recs = []
for _, row in df.iterrows():
    r = recommend_resources(
        impact_score     = row['impact_score'],
        closure_prob     = row['closure_prob'],
        predicted_hours  = row['predicted_duration_hrs'],
        event_cause      = categorize_cause(row['event_cause']),
        is_planned       = row['is_planned']
    )
    recs.append(r)

recs_df = pd.DataFrame(recs)
df = pd.concat([df.reset_index(drop=True), recs_df], axis=1)

print('✅ Full dataset scored and recommendations generated.')

# Show top-10 highest impact
DISP_COLS = ['id','event_type','event_cause','zone','junction',
             'impact_score','closure_prob','predicted_duration_hrs',
             'alert_level','manpower','barricades','diversion_routes']
top_10 = df.sort_values('impact_score', ascending=False)[DISP_COLS].head(10)
top_10 = top_10.round({'impact_score':2, 'closure_prob':2, 'predicted_duration_hrs':1})
print('\n🔴 Top 10 Highest Impact Events:')
top_10

## 10. Post-Event Learning Analytics

In [ ]:
# Compare predicted vs actual duration where actual is known
df_eval = df[(df['duration_hours'] > 0) & (df['duration_hours'] < 200)].copy()

df_eval['error_hrs'] = df_eval['predicted_duration_hrs'] - df_eval['duration_hours']
df_eval['abs_error'] = df_eval['error_hrs'].abs()

print('=== Post-Event Prediction Accuracy ===')
print(f'Mean Absolute Error : {df_eval["abs_error"].mean():.2f} hours')
print(f'Median Abs Error    : {df_eval["abs_error"].median():.2f} hours')
print(f'% within 1 hr       : {(df_eval["abs_error"] <= 1).mean()*100:.1f}%')
print(f'% within 3 hrs      : {(df_eval["abs_error"] <= 3).mean()*100:.1f}%')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Error distribution
df_eval['error_hrs'].clip(-20, 20).hist(bins=40, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].axvline(0, color='red', lw=1.5)
axes[0].set_title('Prediction Error Distribution (clipped ±20 hrs)')
axes[0].set_xlabel('Predicted - Actual (hours)')

# Error by cause
cause_error = df_eval.groupby('event_cause')['abs_error'].median().sort_values()
cause_error.plot(kind='barh', ax=axes[1], color='tomato')
axes[1].set_title('Median Absolute Error by Cause')

plt.tight_layout()
plt.show()

In [ ]:
# Historical accuracy by zone — where does the model struggle?
zone_acc = df_eval.groupby('zone').agg(
    median_actual_hrs=('duration_hours', 'median'),
    median_predicted_hrs=('predicted_duration_hrs', 'median'),
    mae=('abs_error', 'mean'),
    n_events=('id', 'count')
).sort_values('mae', ascending=False).round(2)

print('\n=== Zone-Level Model Accuracy ===')
print(zone_acc)

## 11. Export & Summary

In [ ]:
# Export scored dataset
output_cols = [
    'id', 'event_type', 'event_cause', 'latitude', 'longitude',
    'address', 'zone', 'junction', 'corridor', 'priority',
    'start_datetime', 'status',
    'impact_score', 'closure_prob', 'predicted_duration_hrs',
    'alert_level', 'manpower', 'barricades', 'diversion_routes',
    'patrol_freq_min', 'action_notes'
]

out_df = df[output_cols].copy()
out_df['action_notes'] = out_df['action_notes'].apply(lambda x: ' | '.join(x) if isinstance(x, list) else x)

OUT_PATH = 'event_congestion_scored_output.csv'
out_df.to_csv(OUT_PATH, index=False)
print(f'✅ Scored output saved to: {OUT_PATH}')

# Summary
print('\n=== Final Summary ===')
print(f'Total incidents processed      : {len(df)}')
print(f'Critical alert events          : {(df["alert_level"]=="CRITICAL").sum()}')
print(f'High alert events              : {(df["alert_level"]=="HIGH").sum()}')
print(f'Predicted road closures        : {(df["closure_prob"]>0.5).sum()}')
print(f'Average predicted duration     : {df["predicted_duration_hrs"].mean():.1f} hrs')
print(f'Average manpower recommended   : {df["manpower"].mean():.1f}')
print(f'\nTop corridor by impact        : {df.groupby("corridor")["impact_score"].mean().sort_values().tail(1).index[0]}')
print(f'Top zone by impact             : {df.groupby("zone")["impact_score"].mean().sort_values().tail(1).index[0]}')

## 12. Quick Inference Demo
Simulate a new incoming event and get instant recommendations.

In [ ]:
def predict_event(event_dict):
    """
    Single-event inference demo.
    Pass a dict with keys matching FEAT_DUR / FEAT_CLS.
    """
    row = pd.DataFrame([event_dict])
    dur_pred = np.expm1(lgb_model.predict(row[FEAT_DUR]))[0]
    clos_prob = xgb_model.predict_proba(row[FEAT_CLS])[0, 1]

    dur_s = min(dur_pred / 20, 10)
    clos_s = clos_prob * 10
    pri_b = event_dict['priority_num'] * 1.5
    hot_s = min(event_dict['junction_hotspot_score'] / 20, 2.0)
    impact = min(0.35*dur_s + 0.35*clos_s + 0.15*pri_b + 0.15*hot_s, 10)

    rec = recommend_resources(
        impact_score=impact, closure_prob=clos_prob,
        predicted_hours=dur_pred,
        event_cause=event_dict.get('cause_cat', 'others'),
        is_planned=bool(event_dict.get('is_planned', 0))
    )

    print(f'🔮 Predicted Duration    : {dur_pred:.1f} hours')
    print(f'🚧 Road Closure Prob     : {clos_prob:.2%}')
    print(f'📊 Impact Score          : {impact:.2f} / 10')
    print(f'🚨 Alert Level           : {rec["alert_level"]}')
    print(f'👮 Manpower Recommended  : {rec["manpower"]}')
    print(f'🔴 Barricades            : {rec["barricades"]}')
    print(f'↗️  Diversion Routes      : {rec["diversion_routes"]}')
    print(f'⏱️  Patrol Frequency      : every {rec["patrol_freq_min"]} min')
    print('📋 Action Notes:')
    for n in rec['action_notes']:
        print(f'   {n}')


# Example: Planned public procession on Mysore Road at 8 PM on a Friday
new_event = {
    'hour': 20,
    'dow_num': 4,            # Friday
    'month': 3,
    'is_weekend': 0,
    'is_peak_am': 0,
    'is_peak_pm': 1,
    'is_nighttime': 0,
    'dist_from_centre': 8.5,
    'is_corridor': 1,
    'has_junction': 1,
    'is_planned': 1,
    'priority_num': 1,
    'junction_hotspot_score': 43,
    'zone_risk_score': 433,
    'event_cause_enc': le_dict['event_cause'].transform(['public_event'])[0],
    'corridor_enc': le_dict['corridor'].transform(['Mysore Road'])[0],
    'zone_enc': le_dict['zone'].transform(['West Zone 1'])[0],
    'cause_category_enc': le_dict['cause_category'].transform(['public_event'])[0],
    'latitude': 12.96,
    'longitude': 77.55,
    'cause_cat': 'public_event'
}

print('=== NEW EVENT PREDICTION ===')
print('Planned public procession | Mysore Road | Friday 8 PM\n')
predict_event(new_event)

---
## ✅ Notebook Complete

| Component | Method | Purpose |
|-----------|--------|---------|
| Duration Regression | LightGBM | How long will disruption last? |
| Road Closure Classification | XGBoost | Will road need to be closed? |
| Impact Score | Weighted ensemble | Unified severity score 0–10 |
| Resource Recommender | Rule-based heuristic | Manpower / barricades / diversions |
| Post-event Analytics | Error analysis by zone/cause | Continuous learning loop |

See `APPROACH.md` for full methodology details.